# 11 - Multi-Target ClinicalTrials.gov Exploration

This notebook extends the multi-target pipeline into ClinicalTrials.gov trial evidence.

Input:

```text
data/processed/multi_target_drug_recommendations.csv
```

Outputs:

```text
data/raw/clinicaltrials/multi_target_clinical_trials_raw.json
data/processed/multi_target_clinical_trials.csv
data/processed/multi_target_clinical_trials_summary.csv
data/processed/multi_target_clinical_trials_coverage_summary.csv
```

The notebook is exploratory and limits the number of selected drugs per target. The goal is to verify that trial evidence can be collected consistently across all targets before scaling further.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import re
import time

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "clinicaltrials"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw ClinicalTrials.gov folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

## 2. Load Multi-Target Drug Recommendations

In [ ]:
recommendations_file = PROCESSED_DIR / "multi_target_drug_recommendations.csv"

if not recommendations_file.exists():
    raise FileNotFoundError(
        f"Missing {recommendations_file}. Run notebooks/09_multi_target_chembl_exploration.ipynb first."
    )

recommendations_df = pd.read_csv(recommendations_file)

print("Rows:", len(recommendations_df))
print("Targets:", recommendations_df["target_symbol"].nunique())
print(recommendations_df.groupby("target_symbol").size().sort_values(ascending=False).to_string())

display(recommendations_df.head(20))

## 3. Select Drugs For Exploration

Use the same practical strategy as PubMed:

1. Prefer approved drugs.
2. Then include highest phase investigational drugs.
3. Limit each target to `MAX_DRUGS_PER_TARGET`.

In [ ]:
MAX_DRUGS_PER_TARGET = 8


def phase_value(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


selection_df = recommendations_df.copy()
selection_df["phase_sort"] = selection_df["max_phase"].apply(phase_value)
selection_df["is_approved"] = selection_df["approval_status"].eq("Approved")

selection_df = selection_df.sort_values(
    ["target_symbol", "is_approved", "phase_sort", "drug_name"],
    ascending=[True, False, False, True],
)

drugs_to_search_df = (
    selection_df
    .groupby("target_symbol", group_keys=False)
    .head(MAX_DRUGS_PER_TARGET)
    .reset_index(drop=True)
)

print("Drug-target pairs selected:", len(drugs_to_search_df))
print(drugs_to_search_df.groupby("target_symbol").size().to_string())

display(
    drugs_to_search_df[
        ["target_symbol", "target_display_name", "drug_name", "approval_status", "max_phase", "mechanism_of_action"]
    ]
)

## 4. ClinicalTrials.gov Helper Functions

In [ ]:
CLINICAL_TRIALS_URL = "https://clinicaltrials.gov/api/v2/studies"


def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


def save_json(path: Path, data) -> None:
    with path.open("w") as f:
        json.dump(data, f, indent=2)


def clinical_trials_get(params, retries=4, pause=1.0):
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(CLINICAL_TRIALS_URL, params=params, timeout=(10, 60))
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as error:
            print(f"Attempt {attempt}/{retries} failed: {error}")
            if attempt == retries:
                raise
            time.sleep(pause * attempt)


def safe_join(value):
    if isinstance(value, list):
        return " | ".join(str(item) for item in value)
    if value is None:
        return ""
    return str(value)


def clean_drug_query_name(drug_name: str) -> str:
    value = str(drug_name)
    value = re.sub(r"\b(HYDROCHLORIDE|MESYLATE|SUCCINATE|ANHYDROUS|DIHYDROCHLORIDE|DIMALEATE|MALEATE|TOSYLATE)\b", "", value, flags=re.I)
    value = re.sub(r"\s+", " ", value).strip()
    return value or str(drug_name)


def build_clinical_trial_query(row) -> str:
    drug = clean_drug_query_name(row["drug_name"])
    target_symbol = str(row["target_symbol"])
    target_display = str(row["target_display_name"])

    if target_display and target_display != target_symbol:
        return f"{drug} ({target_symbol} OR {target_display})"
    return f"{drug} {target_symbol}"


def extract_interventions(arms_interventions_module):
    interventions = arms_interventions_module.get("interventions", []) or []
    names = []
    for intervention in interventions:
        name = intervention.get("name")
        intervention_type = intervention.get("type")
        if name and intervention_type:
            names.append(f"{name} ({intervention_type})")
        elif name:
            names.append(name)
    return " | ".join(names)


def extract_trial_record(study, row, query, total_count):
    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    conditions = protocol.get("conditionsModule", {})
    design = protocol.get("designModule", {})
    arms = protocol.get("armsInterventionsModule", {})
    description = protocol.get("descriptionModule", {})

    nct_id = identification.get("nctId")

    return {
        "target_symbol": row["target_symbol"],
        "target_display_name": row["target_display_name"],
        "target_full_name": row["target_full_name"],
        "drug_name": row["drug_name"],
        "molecule_chembl_id": row["molecule_chembl_id"],
        "approval_status": row["approval_status"],
        "query": query,
        "total_matches_for_query": total_count,
        "nct_id": nct_id,
        "brief_title": identification.get("briefTitle"),
        "official_title": identification.get("officialTitle"),
        "overall_status": status.get("overallStatus"),
        "start_date": status.get("startDateStruct", {}).get("date"),
        "completion_date": status.get("completionDateStruct", {}).get("date"),
        "study_type": design.get("studyType"),
        "phases": safe_join(design.get("phases")),
        "conditions": safe_join(conditions.get("conditions")),
        "interventions": extract_interventions(arms),
        "brief_summary": description.get("briefSummary"),
        "url": f"https://clinicaltrials.gov/study/{nct_id}" if nct_id else None,
        "source": "ClinicalTrials.gov",
    }

## 5. Test One Query

In [ ]:
test_row = drugs_to_search_df.iloc[0]
test_query = build_clinical_trial_query(test_row)

test_params = {
    "format": "json",
    "query.term": test_query,
    "pageSize": 5,
    "countTotal": "true",
}

test_data = clinical_trials_get(test_params)

print("Target:", test_row["target_symbol"])
print("Drug:", test_row["drug_name"])
print("Query:", test_query)
print("Total count:", test_data.get("totalCount"))
print("Studies returned:", len(test_data.get("studies", [])))

## 6. Search ClinicalTrials.gov

In [ ]:
PAGE_SIZE = 10
API_PAUSE_SECONDS = 0.35

trial_records = []
raw_payload = []

for _, row in drugs_to_search_df.iterrows():
    query = build_clinical_trial_query(row)
    params = {
        "format": "json",
        "query.term": query,
        "pageSize": PAGE_SIZE,
        "countTotal": "true",
    }

    data = clinical_trials_get(params)
    total_count = int(data.get("totalCount", 0) or 0)
    studies = data.get("studies", []) or []

    raw_payload.append(
        {
            "target_symbol": row["target_symbol"],
            "drug_name": row["drug_name"],
            "query": query,
            "response": data,
        }
    )

    print(f"{row['target_symbol']} | {row['drug_name']}: {total_count} total, fetched {len(studies)}")

    for study in studies:
        trial_records.append(extract_trial_record(study, row, query, total_count))

    time.sleep(API_PAUSE_SECONDS)

raw_trials_file = RAW_DIR / "multi_target_clinical_trials_raw.json"
save_json(raw_trials_file, raw_payload)

clinical_trials_df = pd.DataFrame(trial_records)

print("Saved raw trials:", raw_trials_file)
print("Trial rows collected:", len(clinical_trials_df))

display(clinical_trials_df.head(20))

## 7. Clean And Save Trial Records

In [ ]:
expected_columns = [
    "target_symbol",
    "target_display_name",
    "target_full_name",
    "drug_name",
    "molecule_chembl_id",
    "approval_status",
    "query",
    "total_matches_for_query",
    "nct_id",
    "brief_title",
    "official_title",
    "overall_status",
    "start_date",
    "completion_date",
    "study_type",
    "phases",
    "conditions",
    "interventions",
    "brief_summary",
    "url",
    "source",
]

if clinical_trials_df.empty:
    clean_clinical_trials_df = pd.DataFrame(columns=expected_columns)
else:
    clean_clinical_trials_df = (
        clinical_trials_df
        .drop_duplicates(subset=["target_symbol", "drug_name", "nct_id"])
        .reset_index(drop=True)
    )

clinical_trials_file = PROCESSED_DIR / "multi_target_clinical_trials.csv"
clean_clinical_trials_df.to_csv(clinical_trials_file, index=False)

print("Rows before cleaning:", len(clinical_trials_df))
print("Rows after cleaning:", len(clean_clinical_trials_df))
print("Saved:", clinical_trials_file)

display(clean_clinical_trials_df.head(20))

## 8. Build Clinical Trial Summary

In [ ]:
ACTIVE_STATUSES = {
    "RECRUITING",
    "NOT_YET_RECRUITING",
    "ACTIVE_NOT_RECRUITING",
    "ENROLLING_BY_INVITATION",
}


def count_active_trials(status_series):
    return status_series.fillna("").str.upper().isin(ACTIVE_STATUSES).sum()


def count_late_phase_trials(phase_series):
    count = 0
    for phase_text in phase_series.dropna():
        text = str(phase_text).upper().replace("_", " ")
        if "PHASE 3" in text or "PHASE3" in text or "PHASE 4" in text or "PHASE4" in text:
            count += 1
    return count


def clinical_evidence_score(row):
    score = 0.0
    if row["clinical_trial_count"] > 0:
        score += 0.4
    if row["active_trial_count"] > 0:
        score += 0.2
    if row["late_phase_trial_count"] > 0:
        score += 0.3
    if row["clinical_trial_count"] >= 5:
        score += 0.1
    return round(min(score, 1.0), 2)


summary_columns = [
    "target_symbol",
    "target_display_name",
    "drug_name",
    "molecule_chembl_id",
    "clinical_trial_count",
    "active_trial_count",
    "late_phase_trial_count",
    "top_trial_titles",
    "clinical_evidence_score",
]

if clean_clinical_trials_df.empty:
    clinical_trials_summary_df = pd.DataFrame(columns=summary_columns)
else:
    clinical_trials_summary_df = (
        clean_clinical_trials_df
        .groupby(["target_symbol", "target_display_name", "drug_name", "molecule_chembl_id"], dropna=False)
        .agg(
            clinical_trial_count=("nct_id", "nunique"),
            active_trial_count=("overall_status", count_active_trials),
            late_phase_trial_count=("phases", count_late_phase_trials),
            top_trial_titles=("brief_title", lambda values: " | ".join(list(values.dropna())[:3])),
        )
        .reset_index()
    )
    clinical_trials_summary_df["clinical_evidence_score"] = clinical_trials_summary_df.apply(
        clinical_evidence_score,
        axis=1,
    )

clinical_trials_summary_file = PROCESSED_DIR / "multi_target_clinical_trials_summary.csv"
clinical_trials_summary_df.to_csv(clinical_trials_summary_file, index=False)

print("Saved:", clinical_trials_summary_file)
print("Rows:", len(clinical_trials_summary_df))

display(clinical_trials_summary_df.head(30))

## 9. Coverage Summary

In [ ]:
coverage_rows = []

for target_symbol, target_df in drugs_to_search_df.groupby("target_symbol"):
    trials_target_df = clean_clinical_trials_df[clean_clinical_trials_df["target_symbol"] == target_symbol]
    searched_count = target_df["drug_name"].nunique()
    matched_count = trials_target_df["drug_name"].nunique() if not trials_target_df.empty else 0

    coverage_rows.append(
        {
            "target_symbol": target_symbol,
            "target_display_name": target_df["target_display_name"].iloc[0],
            "source": "ClinicalTrials.gov",
            "drug_pairs_searched": int(searched_count),
            "drugs_with_trial_evidence": int(matched_count),
            "clinical_trial_rows": int(len(trials_target_df)),
            "raw_saved": True,
            "processed_saved": True,
            "status": "working" if matched_count > 0 else "needs_review",
            "notes": "" if matched_count > 0 else "No ClinicalTrials.gov matches found for selected drugs.",
        }
    )

clinical_trials_coverage_df = pd.DataFrame(coverage_rows)

clinical_trials_coverage_file = PROCESSED_DIR / "multi_target_clinical_trials_coverage_summary.csv"
clinical_trials_coverage_df.to_csv(clinical_trials_coverage_file, index=False)

print("Saved:", clinical_trials_coverage_file)
display(clinical_trials_coverage_df)

## 10. Final Summary

In [ ]:
print("Multi-Target ClinicalTrials.gov Exploration Complete")
print("=" * 80)
print("Drug-target pairs searched:", len(drugs_to_search_df))
print("Trial rows:", len(clean_clinical_trials_df))
print("Summary rows:", len(clinical_trials_summary_df))

print("\nTrial rows by target:")
if not clean_clinical_trials_df.empty:
    print(clean_clinical_trials_df.groupby("target_symbol").size().sort_values(ascending=False).to_string())
else:
    print("No ClinicalTrials.gov rows found.")

print("\nFiles created:")
for path in [
    raw_trials_file,
    clinical_trials_file,
    clinical_trials_summary_file,
    clinical_trials_coverage_file,
]:
    print("-", path)

display(clinical_trials_coverage_df)